Esta celda instala las librerías necesarias para ejecutar ViZDoom en Kaggle. Incluye dependencias del sistema (Boost, SDL2, OpenAL, FFmpeg) y las versiones correctas de vizdoom, gym, pyvirtualdisplay, imageio y PyTorch para asegurar compatibilidad y permitir el entrenamiento y visualización del entorno.

In [ ]:
!apt-get update -y && apt-get install -y libboost-all-dev cmake libsdl2-dev libopenal-dev ffmpeg
!pip install vizdoom[gym] gym==0.26.5 pyvirtualdisplay==3.0.0 imageio==2.27.0
!pip install torch torchvision

En esta celda importamos todas las librerías necesarias para entrenar un agente de aprendizaje por refuerzo en VizDoom. Se incluyen herramientas para manejo del entorno (gym, imageio), procesamiento de imágenes (PIL), cálculo numérico (numpy) y construcción de redes neuronales (PyTorch).
Finalmente, se fija una semilla aleatoria (SEED = 42) para asegurar reproducibilidad en los resultados del entrenamiento.

In [ ]:
import os
import gym
import time
import random
import numpy as np
from collections import deque
from PIL import Image
import imageio
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


Esta celda actualiza los repositorios de Ubuntu e instala todas las dependencias del sistema necesarias para que VizDoom pueda compilar y funcionar correctamente en Kaggle, incluyendo librerías gráficas, de sonido y herramientas de construcción como cmake y ffmpeg.

In [ ]:
!apt-get update -y
!apt-get install -y \
    cmake \
    libboost-all-dev \
    libsdl2-dev \
    libopenal-dev \
    libjpeg-dev \
    libbz2-dev \
    zlib1g-dev \
    libfluidsynth-dev \
    libgme-dev \
    libopenal-dev \
    timidity \
    ffmpeg


Esta celda instala todas las dependencias necesarias para compilar e instalar ViZDoom desde cero en Google Colab/Kaggle.
Incluye librerías del sistema (audio, video, compresión, SDL2) y herramientas como cmake para construir el motor Doom.

In [ ]:
!git clone https://github.com/mwydmuch/ViZDoom.git
%cd ViZDoom
!git checkout master

# Compilar librería base
!cmake -DCMAKE_BUILD_TYPE=Release .
!make -j4

# Instalar módulo de Python
%cd python
!pip install .


Esta celda crea una clase personalizada de entorno Gym para VizDoom. Configura el juego, procesa las imágenes, define las acciones permitidas, administra el frame skip y el frame stacking, y permite que el agente interactúe con Doom como si fuera un entorno estándar de Reinforcement Learning.

In [ ]:
import gym
import numpy as np
from collections import deque
from PIL import Image

from vizdoom import DoomGame, Mode, ScreenFormat, ScreenResolution


class VizdoomGymEnv(gym.Env):
    def __init__(self, config_path, frame_skip=4, stack_frames=4, resolution=(320,240), grayscale=True):
        super().__init__()
        
        # ---------------------------
        # Crear juego
        # ---------------------------
        self.game = DoomGame()
        self.game.load_config(config_path)
        self.game.set_window_visible(False)
        self.game.set_sound_enabled(False)
        self.game.set_mode(Mode.PLAYER)

        # configurar pantalla
        self.game.set_screen_format(ScreenFormat.RGB24)
        self.game.set_screen_resolution(ScreenResolution.RES_320X240)

        # *** LA LÍNEA QUE TE FALTABA ***
        self.game.init()

        # ---------------------------
        # Parámetros
        # ---------------------------
        self.frame_skip = frame_skip
        self.stack_frames = stack_frames
        self.grayscale = grayscale
        self.resolution = resolution
        
        # Acciones combinadas (8 botones disponibles)
        self.actions = [
            [1,0,0,0,0,0,0,0], # forward
            [0,1,0,0,0,0,0,0], # backward
            [0,0,1,0,0,0,0,0], # turn left
            [0,0,0,1,0,0,0,0], # turn right
            [0,0,0,0,1,0,0,0], # strafe left
            [0,0,0,0,0,1,0,0], # strafe right
            [0,0,0,0,0,0,1,0], # jump
            [0,0,0,0,0,0,0,1], # attack
            [1,0,0,0,0,0,0,1], # forward + shoot
            [0,0,0,1,0,0,0,1], # turn right + shoot
        ]
        
        self.action_space = gym.spaces.Discrete(len(self.actions))

        obs_shape = (resolution[1], resolution[0], 1 if grayscale else 3)
        self.observation_space = gym.spaces.Box(
            0, 255, obs_shape, dtype=np.uint8
        )

        # frame stack
        self._frame_stack = deque(maxlen=stack_frames)

    # ---------------------------------------------------------
    # reset
    # ---------------------------------------------------------
    def reset(self):
        self.game.new_episode()

        state = self.game.get_state()
        if state is None:
            # seguridad extra
            frame = np.zeros((self.observation_space.shape), dtype=np.uint8)
        else:
            raw = state.screen_buffer
            frame = self._process_frame(raw)

        self._frame_stack.clear()
        for _ in range(self.stack_frames):
            self._frame_stack.append(frame)

        return np.stack(self._frame_stack, axis=2)

    # ---------------------------------------------------------
    # step
    # ---------------------------------------------------------
    def step(self, action_idx):
        action = self.actions[action_idx]
        reward = 0
        done = False

        for _ in range(self.frame_skip):
            reward += self.game.make_action(action)
            done = self.game.is_episode_finished()
            if done:
                break

        if done:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
        else:
            raw = self.game.get_state().screen_buffer
            frame = self._process_frame(raw)
            self._frame_stack.append(frame)
            obs = np.stack(self._frame_stack, axis=2)

        return obs, reward, done, {}

    # ---------------------------------------------------------
    # Procesar imagen
    # ---------------------------------------------------------
    def _process_frame(self, frame):
        img = Image.fromarray(frame)
        if self.grayscale:
            img = img.convert("L")
        img = img.resize(self.resolution)
        arr = np.array(img, dtype=np.uint8)

        if self.grayscale:
            arr = arr[..., np.newaxis]
        return arr

    # ---------------------------------------------------------
    # Render
    # ---------------------------------------------------------
    def render(self, mode="rgb_array"):
        if self.game.is_episode_finished():
            return np.zeros(self.observation_space.shape, dtype=np.uint8)
        return self.game.get_state().screen_buffer

    # ---------------------------------------------------------
    # close
    # ---------------------------------------------------------
    def close(self):
        self.game.close()


**PreprocessFrame**

Toma una imagen del entorno.

La convierte a escala de grises.

La redimensiona a 84×84 píxeles.

Devuelve la imagen lista para usar en la red neuronal.

**FrameStack**

Guarda las últimas k imágenes (frames).

En reset(), duplica el primer frame para llenar el stack inicial.

En append(), agrega el nuevo frame y devuelve un tensor con los últimos k frames apilados.

Esto permite que el agente vea movimiento, no solo imágenes sueltas.

In [ ]:
class PreprocessFrame:
    def __init__(self, shape=(84,84)):
        self.shape = shape

    def __call__(self, obs):
        img = Image.fromarray(obs.squeeze())
        img = img.convert("L").resize(self.shape)
        return np.array(img, dtype=np.uint8)

class FrameStack:
    def __init__(self, k):
        self.k = k
        self.frames = deque(maxlen=k)

    def reset(self, frame):
        for _ in range(self.k):
            self.frames.append(frame)
        return np.stack(self.frames, axis=2)

    def append(self, frame):
        self.frames.append(frame)
        return np.stack(self.frames, axis=2)


**Convoluciones (self.conv)**

Extraen características de las imágenes (bordes, formas, enemigos, etc.).
Son 3 capas conv + ReLU.

**Cálculo de tamaño (dummy)**

Se usa una imagen falsa de prueba para saber cuántas neuronas salen de las convoluciones.

**Capa fully-connected (self.fc)**

Reduce las características a un vector de 512 valores.

**policy_logits**

Produce la probabilidad de cada acción que el agente puede tomar.

value_head
Predice el "valor" del estado, usado por PPO para aprender.

**forward(x)**

Normaliza la imagen (0–255 → 0–1).

Extrae características.

Pasa por la red.

In [ ]:
class CNNBase(nn.Module):
    def __init__(self, input_channels, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_channels, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, 84, 84)
            conv_out = self.conv(dummy).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512),
            nn.ReLU()
        )

        self.policy_logits = nn.Linear(512, n_actions)
        self.value_head = nn.Linear(512, 1)

    def forward(self, x):
        x = x / 255.0
        features = self.conv(x)
        features = self.fc(features)
        return self.policy_logits(features), self.value_head(features).squeeze(-1)


**RolloutBuffer**

Es una clase que guarda todas las experiencias que el agente recolecta durante un rollout de PPO:

states: estados observados

actions: acciones tomadas

logprobs: log-probabilidades de esas acciones

rewards: recompensas obtenidas

dones: si el episodio terminó

values: estimaciones del valor del estado hechas por la red

La función clear() simplemente reinicia el buffer.

**compute_gae()**

Calcula el GAE (Generalized Advantage Estimation), una técnica usada en PPO para estimar:

Ventajas (advantages) → qué tan buena fue una acción comparada con lo esperado.

Retornos (returns) → suma de recompensas futuras corregidas con el valor del estado.

Esto ayuda a que el entrenamiento sea más estable y eficiente, reduciendo el ruido en los gradientes.

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.dones = []
        self.values = []

    def clear(self):
        self.__init__()

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    values = values + [0]

    for step in reversed(range(len(rewards))):
        delta = rewards[step] + gamma * values[step+1] * (1-dones[step]) - values[step]
        gae = delta + gamma * lam * (1-dones[step]) * gae
        advantages.insert(0, gae)

    returns = [adv + val for adv,val in zip(advantages, values[:-1])]
    return advantages, returns


Este código implementa la parte central del algoritmo PPO (Proximal Policy Optimization), que entrena a la red neuronal usando los datos recolectados durante los episodios.

**¿Qué hace esta clase?**

Recibe la red (actor-critic) y los hiperparámetros del método.

Calcula la actualización de la política y del valor usando PPO.

Optimiza la red neuronal para que tome mejores decisiones.

**Partes importantes**

1. Constructor (__init__)

Inicializa:

La red y el optimizador (Adam).

El clipping de PPO (clip_eps).

Número de epochs, tamaño de batch.

Pesos de las pérdidas (valor y entropía).

El dispositivo (CPU o GPU).

En resumen: configura el entrenador.

2. Método update(buffer)

Actualiza la red usando los datos recolectados:

Convierte los datos del buffer a tensores:

estados

acciones

logprobs antiguos

valores y recompensas

Calcula ventajas y retornos usando GAE
(indica qué tan buenas fueron las acciones realmente).

**Normaliza ventajas**
→ esto mejora la estabilidad del entrenamiento.

Entrena por múltiple epochs y batches:

Calcula nueva política y valores.

Calcula el ratio entre prob nueva / prob vieja.

Aplica el clipping PPO para evitar cambios bruscos.

**Calcula:**

loss de política

loss de valor

entropía (exploración)

Backpropagation + gradient clipping
para evitar explosión de gradientes.

In [ ]:
class PPO:
    def __init__(self, net, lr=2.5e-4, clip_eps=0.2, epochs=4,
                 batch_size=64, value_coef=0.5, ent_coef=0.01, device="cpu"):
        self.net = net.to(device)
        self.optimizer = optim.Adam(self.net.parameters(), lr=lr)
        self.clip_eps = clip_eps
        self.epochs = epochs
        self.batch_size = batch_size
        self.value_coef = value_coef
        self.ent_coef = ent_coef
        self.device = device

    def update(self, buffer):
        states = torch.tensor(np.stack(buffer.states), dtype=torch.float32).to(self.device)
        actions = torch.tensor(buffer.actions, dtype=torch.int64).to(self.device)
        old_logprobs = torch.tensor(buffer.logprobs, dtype=torch.float32).to(self.device)

        advantages, returns = compute_gae(buffer.rewards, buffer.values, buffer.dones)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(self.device)
        returns = torch.tensor(returns, dtype=torch.float32).to(self.device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        dataset_size = len(states)

        for _ in range(self.epochs):
            idxs = np.arange(dataset_size)
            np.random.shuffle(idxs)

            for start in range(0, dataset_size, self.batch_size):
                batch_idx = idxs[start:start+self.batch_size]

                b_states = states[batch_idx].permute(0,3,1,2)
                logits, vals = self.net(b_states)
                probs = F.softmax(logits, dim=-1)
                dist = torch.distributions.Categorical(probs)

                new_logprobs = dist.log_prob(actions[batch_idx])
                ratio = torch.exp(new_logprobs - old_logprobs[batch_idx])

                surr1 = ratio * advantages[batch_idx]
                surr2 = torch.clamp(ratio, 1-self.clip_eps, 1+self.clip_eps) * advantages[batch_idx]

                policy_loss = -torch.min(surr1, surr2).mean()
                value_loss = F.mse_loss(vals, returns[batch_idx])
                entropy = dist.entropy().mean()

                loss = policy_loss + self.value_coef * value_loss - self.ent_coef * entropy

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
                self.optimizer.step()


La función entrena al agente PPO en Doom siguiendo estos pasos:

**Prepara todo**

Procesa las imágenes (84×84).

Apila 4 frames para dar memoria al agente.

Crea el buffer donde se guardan experiencias.

**Bucle principal de entrenamiento**

Mientras no se llegue al número total de timesteps:

Limpia el buffer.

Ejecuta acciones durante rollout_length pasos.

Para cada paso:

El agente predice la acción y el valor.

Se ejecuta la acción en Doom.

Se guarda estado, acción, recompensa, done y valor.

Si la partida termina, se reinicia el entorno.

**Actualiza el PPO**

Con los datos del buffer, el agente aprende (backpropagation).

**Guarda el modelo**

Cada ciclo guarda ppo_doom.pth.

**Imprime progreso**

Muestra timesteps, episodios y avisos de guardado.

In [ ]:
def train(env, agent, ppo, total_timesteps=1_000_000, rollout_length=2048,
          device="cpu", save_path="ppo_doom.pth"):

    obs_proc = PreprocessFrame((84,84))
    frame_stack = FrameStack(4)
    buffer = RolloutBuffer()

    timestep = 0
    episode = 0

    obs = env.reset()
    proc = obs_proc(obs)
    state = frame_stack.reset(proc)

    while timestep < total_timesteps:
        buffer.clear()

        for _ in range(rollout_length):
            st_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).permute(0,3,1,2).to(device)
            logits, value = agent(st_tensor)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)

            action = dist.sample().item()
            logp = dist.log_prob(torch.tensor(action).to(device)).item()

            next_obs, reward, done, info = env.step(action)
            proc = obs_proc(next_obs)
            next_state = frame_stack.append(proc)

            buffer.states.append(state)
            buffer.actions.append(action)
            buffer.logprobs.append(logp)
            buffer.rewards.append(reward)
            buffer.dones.append(done)
            buffer.values.append(value.item())

            state = next_state
            timestep += 1

            if done:
                episode += 1
                obs = env.reset()
                proc = obs_proc(obs)
                state = frame_stack.reset(proc)

        ppo.update(buffer)
        torch.save(agent.state_dict(), save_path)

        print(f"Timestep {timestep} — Episode {episode} — Guardado {save_path}")


In [ ]:
import cv2
import torch
import numpy as np


# ============================================================
# MANEJO SEGURO DEL FRAME
# ============================================================

def get_frame(obs):
    if isinstance(obs, dict):
        return fix_frame_gray(obs["screen"])
    if isinstance(obs, np.ndarray):
        return fix_frame_gray(obs)
    raise ValueError("Obs no reconocida")



# ============================================================
# CONVERTIR GRIS -> BGR PARA VIDEO
# ============================================================

def gray_to_bgr(frame):
    if frame.ndim == 2:
        return cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    return frame

def to_bgr(frame):
    """Recibe frame gris y devuelve un frame 3 canales para VideoWriter."""
    return cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

# ============================================================
# PREPROCESS
# ============================================================

def preprocess(frame):
    frame = cv2.resize(frame, (84, 84))
    frame = frame / 255.0
    return torch.tensor(frame, dtype=torch.float32).unsqueeze(0)

In [ ]:
def fix_frame_gray(frame):
    f = frame

    # Caso 4D: (H, W, 4, 1) → (H, W, 4)
    if f.ndim == 4 and f.shape[-1] == 1:
        f = np.squeeze(f, axis=-1)

    # Si tiene 4 canales (stack), usar solo el primero
    if f.ndim == 3 and f.shape[2] == 4:
        f = f[:, :, 0]

    # Si ya es gris (H, W, 1) → (H, W)
    if f.ndim == 3 and f.shape[2] == 1:
        f = np.squeeze(f, axis=-1)

    # Si es RGB → convertir a gris
    if f.ndim == 3 and f.shape[2] == 3:
        f = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)

    # Si es 2D ya está OK
    if f.ndim == 2:
        return f.astype(np.uint8)

    raise ValueError(f"Formato imposible de convertir: shape={frame.shape}")


In [ ]:
def extract_frame(obs):
    # Caso 1: diccionario estilo VizDoomGym
    if isinstance(obs, dict):
        frame = obs.get("screen", None)
        if frame is None:
            raise ValueError("El diccionario no contiene 'screen'")
        return frame
    
    # Caso 2: tu entorno devuelve directamente la imagen (numpy)
    if isinstance(obs, np.ndarray):
        return obs

    # Caso 3: devuelve tupla (imagen, variables)
    if isinstance(obs, (list, tuple)):
        for item in obs:
            if isinstance(item, np.ndarray):
                return item
    
    raise ValueError("No pude extraer un frame válido de la observación")


evaluate_random prueba el entorno sin inteligencia, usando acciones aleatorias, para ver qué tan difícil es el escenario.

Reinicia el entorno por varios episodios.

En cada paso toma una acción aleatoria.

Suma las recompensas para saber el puntaje total del episodio.

Devuelve una lista con los puntajes obtenidos en cada episodio.

In [ ]:
def evaluate_random(env, episodes=5, save_video_path=None):
    scores = []
    writer = None

    if save_video_path:
        writer = cv2.VideoWriter(
            save_video_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            20,
            (320, 240)
        )

    for ep in range(episodes):
        obs = env.reset()
        frame = get_frame(obs)
        done = False
        ep_reward = 0

        while not done:
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            ep_reward += reward

            frame = get_frame(obs)

            if writer:
                writer.write(to_bgr(frame))

        scores.append(ep_reward)

    if writer:
        writer.release()

    return scores

Esta función prueba tu agente entrenado, usando su red neuronal para decidir las acciones.

Qué hace:

Reinicia el entorno.

Preprocesa y apila 4 frames (como en entrenamiento).

El agente elige la mejor acción (argmax).

Suma la recompensa total del episodio.

Guarda los frames como video.

Devuelve los puntajes obtenidos por el agente.

In [ ]:
def evaluate_agent(env, agent, episodes=5, device="cpu", save_video_path=None):
    scores = []
    writer = None

    if save_video_path:
        writer = cv2.VideoWriter(
            save_video_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            20,
            (320, 240)
        )

    for ep in range(episodes):
        obs = env.reset()
        frame = get_frame(obs)

        done = False
        total_reward = 0

        # Stack de 4 frames
        f = preprocess(frame)
        frame_stack = [f for _ in range(4)]

        while not done:

            stacked = torch.cat(frame_stack, dim=0).unsqueeze(0).to(device)
            with torch.no_grad():
                logits, value = agent(stacked)

            probs = torch.softmax(logits, dim=1)
            action = torch.multinomial(probs, 1).item()

            obs, reward, done, info = env.step(action)
            total_reward += reward

            frame = get_frame(obs)
            new_f = preprocess(frame)

            frame_stack.pop(0)
            frame_stack.append(new_f)

            if writer:
                writer.write(to_bgr(frame))

        scores.append(total_reward)

    if writer:
        writer.release()

    return scores

Selecciona GPU o CPU

Usa CUDA si está disponible para acelerar el entrenamiento.

Carga el escenario de Doom

Abre el archivo basic.cfg para crear el entorno del juego.

Crea el agente

CNNBase: red neuronal que verá 4 frames (stack) y escogerá acciones.

El número de acciones viene del entorno.

Crea el algoritmo PPO

PPO usa la red del agente para aprender a jugar Doom.

Entrena el agente

Corre 500.000 pasos de entrenamiento.

Cada 1024 pasos junta experiencias y actualiza la red.

Guarda el modelo en doom_agent.pth.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

config_path = "/kaggle/working/ViZDoom/scenarios/basic.cfg"  # AJUSTAR

env = VizdoomGymEnv(config_path)
agent = CNNBase(input_channels=4, n_actions=env.action_space.n).to(device)
ppo = PPO(agent, device=device)

train(env, agent, ppo,
      total_timesteps=500_000,
      rollout_length=1024,
      device=device,
      save_path="doom_agent.pth")


In [ ]:
scores_random = evaluate_random(env, episodes=5, save_video_path="random_agent.mp4")
scores_trained = evaluate_agent(env, agent, episodes=5, device=device, save_video_path="trained_agent.mp4")

print("Random scores:", scores_random)
print("Trained scores:", scores_trained)

env.close()
